In [87]:
# Imports
import os
import pandas as pd
import numpy as np
import json
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

print("Imports Done ✔")


Imports Done ✔


# Model A - Visit Risk Classification
**Business Purpose:**
Predict whether a hospital visit represents Low, Medium or High operational and clinical risk

**Target:** `risk_score` - Low/Medium/High

**Split:** Time-based - earliest 80% train, latest 20% test 

**Why time based?** Prevents data leakage from future visits influencing predictions on past visits

In [88]:
# Load dataset

df = pd.read_csv(os.path.join("..", "outputs", "model_table_enriched.csv"), parse_dates=["registration_date", "visit_date", "billing_date"])
print("Shape: ", df.shape)
df.head()

Shape:  (25000, 30)


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_id,visit_date,department,...,risk_numeric,claim_numeric,is_rejected,days_since_registration,visit_frequency,avg_los_per_patient,provider_rejection_rate,visit_month,visit_dayofweek,high_cost_visit_flag
0,2,15,F,Mumbai,CareOne,0,2025-12-27,8,2026-01-01,General,...,0,2,1,5,4,21.120000,0.256876,1,3,0
1,12,3,M,Bangalore,CareOne,0,2025-08-13,65,2026-01-01,ICU,...,2,2,1,141,8,23.750000,0.256876,1,3,1
2,129,44,M,Pune,MediCareX,1,2025-07-20,651,2026-01-01,ICU,...,2,1,0,165,3,32.460000,0.242556,1,3,1
3,133,47,F,Delhi,CareOne,1,2025-11-02,670,2026-01-01,General,...,1,0,0,60,3,30.056667,0.256876,1,3,0
4,139,14,F,Chennai,SecureLife,1,2025-02-05,706,2026-01-01,Cardiology,...,1,0,0,330,9,29.030000,0.157496,1,3,1


In [89]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 30 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   patient_id               25000 non-null  int64         
 1   age                      25000 non-null  int64         
 2   gender                   25000 non-null  object        
 3   city                     25000 non-null  object        
 4   insurance_provider       25000 non-null  object        
 5   chronic_flag             25000 non-null  int64         
 6   registration_date        25000 non-null  datetime64[ns]
 7   visit_id                 25000 non-null  int64         
 8   visit_date               25000 non-null  datetime64[ns]
 9   department               25000 non-null  object        
 10  visit_type               25000 non-null  object        
 11  length_of_stay_hours     25000 non-null  float64       
 12  risk_score               25000 n

In [90]:
# define risk features

risk_target = "risk_score"
risk_features = [
    "age", 
    "gender", 
    "city", 
    "insurance_provider", 
    "chronic_flag", 
    "department", 
    "visit_type", 
    "length_of_stay_hours",
    "days_since_registration", 
    "visit_frequency", 
    "avg_los_per_patient", 
    "visit_month", 
    "visit_dayofweek"
]

risk_df = df[risk_features + [risk_target , "visit_date"]].copy()
risk_df = risk_df.dropna(subset = [risk_target , "visit_date"])
print("Risk Dataset Shape: ", risk_df.shape)
risk_df.head() 

Risk Dataset Shape:  (25000, 15)


,age,gender,city,insurance_provider,chronic_flag,department,visit_type,length_of_stay_hours,days_since_registration,visit_frequency,avg_los_per_patient,visit_month,visit_dayofweek,risk_score,visit_date
0,15,F,Mumbai,CareOne,0,General,OPD,9.63,5,4,21.120000,1,3,Low,2026-01-01
1,3,M,Bangalore,CareOne,0,ICU,ICU,59.60,141,8,23.750000,1,3,High,2026-01-01
2,44,M,Pune,MediCareX,1,ICU,ER,59.28,165,3,32.460000,1,3,High,2026-01-01
3,47,F,Delhi,CareOne,1,General,OPD,25.15,60,3,30.056667,1,3,Medium,2026-01-01
4,14,F,Chennai,SecureLife,1,Cardiology,ER,42.88,330,9,29.030000,1,3,Medium,2026-01-01


In [91]:
# Step 3 - Time Based Split
risk_df = risk_df.sort_values("visit_date").reset_index(drop=True)
split_idx = int(len(risk_df)*0.8)

risk_train = risk_df.iloc[:split_idx].copy()
risk_test = risk_df.iloc[split_idx:].copy()

X_train_risk = risk_train[risk_features]
X_test_risk = risk_test[risk_features]

y_train_risk = risk_train[risk_target]
y_test_risk = risk_test[risk_target]

print(f"Train Shape: {X_train_risk.shape}")
print(f"Test Shape: {X_test_risk.shape}")
print(f"Train Period: {risk_train['visit_date'].min().date()} --> {risk_train['visit_date'].max().date()}")
print(f"Test Period: {risk_test['visit_date'].min().date()} --> {risk_test['visit_date'].max().date()}")

Train Shape: (20000, 13)
Test Shape: (5000, 13)
Train Period: 2025-01-21 --> 2026-01-02
Test Period: 2026-01-02 --> 2026-01-20


# Preprocessing Pipeline

Two transformers: \
    ▪ **Numeric** -> SimpleImputer (median startegy)  \
    ▪ **Categorical** -> SimpleImputer + OneHotEncoder

Combined using ColumnTransformer. This entire pipeline gets saved with the model - no separate preprocessing step at inference time

In [92]:
# Step 4 - Preprocessing Pipeline

risk_numeric_features = [f for f in risk_features if df[f].dtype != "object"]
risk_categorical_features = [f for f in risk_features if f not in risk_numeric_features]
print(f" Numeric features: {risk_numeric_features}")
print(f" Categorical features: {risk_categorical_features}")

 Numeric features: ['age', 'chronic_flag', 'length_of_stay_hours', 'days_since_registration', 'visit_frequency', 'avg_los_per_patient', 'visit_month', 'visit_dayofweek']
 Categorical features: ['gender', 'city', 'insurance_provider', 'department', 'visit_type']


In [93]:
# Applying Imputer

risk_numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

risk_categorical_transformer = Pipeline(
    steps =[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown = "ignore"))
    ]
)

risk_preprocessor = ColumnTransformer(transformers=[
    ("num", risk_numeric_transformer, risk_numeric_features),
    ("cat", risk_categorical_transformer, risk_categorical_features)
])

print("Preprocessing Pipeline Ready ✔")

Preprocessing Pipeline Ready ✔


# Baseline Model : Logistic Regression
We always start with the simplest model.  \
Logistic Regression will give us a baseline to beat. \
`class_weight = "balanced"` will be used to handle class imbalance

In [94]:
# Step 5 - Logistic Regression Model

risk_baseline_model = Pipeline(steps=[
    ("preprocessor", risk_preprocessor),
    ("classifier", LogisticRegression(
        max_iter = 1000,
        class_weight = "balanced"
    ))
])

# Train the model
risk_baseline_model.fit(X_train_risk, y_train_risk)

# Predict
risk_baseline_pred_train = risk_baseline_model.predict(X_train_risk)
risk_baseline_pred_test = risk_baseline_model.predict(X_test_risk)

baseline_acc = accuracy_score(y_test_risk, risk_baseline_pred_test)
print(f"Risk Baseline Train Accuracy :", round(accuracy_score(y_train_risk, risk_baseline_pred_train),4))
print(f"Risk Baseline Test Accuracy :", round(accuracy_score(y_test_risk, risk_baseline_pred_test),4))
print(f"Risk Baseline Test Weighted F1 :", round(f1_score(y_test_risk, risk_baseline_pred_test, average="weighted"),4))

print("Classification Report:\n")
print(classification_report(y_test_risk, risk_baseline_pred_test))
print("Confusion Matrix:\n", confusion_matrix(y_test_risk, risk_baseline_pred_test))

Risk Baseline Train Accuracy : 0.9246
Risk Baseline Test Accuracy : 0.9248
Risk Baseline Test Weighted F1 : 0.9251
Classification Report:

              precision    recall  f1-score   support

        High       0.92      0.95      0.93       838
         Low       0.96      0.93      0.94      2360
      Medium       0.88      0.91      0.90      1802

    accuracy                           0.92      5000
   macro avg       0.92      0.93      0.92      5000
weighted avg       0.93      0.92      0.93      5000

Confusion Matrix:
 [[ 795    0   43]
 [   0 2186  174]
 [  73   86 1643]]


# Advanced Model - Random Forest


In [95]:
# Step 6 - Random Forest Model

risk_rf_model = Pipeline(steps=[
    ("preprocessor", risk_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators = 200,
        max_depth = 8,
        min_samples_split = 20,
        min_samples_leaf = 10,
        class_weight = "balanced_subsample",
        random_state = 42
    ))
])

# Train the model
risk_rf_model.fit(X_train_risk, y_train_risk)

# Predict
risk_rf_pred_train = risk_rf_model.predict(X_train_risk)
risk_rf_pred_test = risk_rf_model.predict(X_test_risk)

rf_acc = accuracy_score(y_test_risk, risk_rf_pred_test)
print(f"Risk RF Train Accuracy :", round(accuracy_score(y_train_risk, risk_rf_pred_train),4))
print(f"Risk RF Test Accuracy :", round(accuracy_score(y_test_risk, risk_rf_pred_test),4))
print(f"Risk RF Test Weighted F1 :", round(f1_score(y_test_risk, risk_rf_pred_test, average="weighted"),4))

print("Classification Report:\n")
print(classification_report(y_test_risk, risk_rf_pred_test))
print("Confusion Matrix:\n", confusion_matrix(y_test_risk, risk_rf_pred_test))

Risk RF Train Accuracy : 0.9389
Risk RF Test Accuracy : 0.9294
Risk RF Test Weighted F1 : 0.9292
Classification Report:

              precision    recall  f1-score   support

        High       0.94      0.93      0.94       838
         Low       0.94      0.96      0.95      2360
      Medium       0.91      0.89      0.90      1802

    accuracy                           0.93      5000
   macro avg       0.93      0.93      0.93      5000
weighted avg       0.93      0.93      0.93      5000

Confusion Matrix:
 [[ 780    0   58]
 [   0 2262   98]
 [  47  150 1605]]


# XGBoost Model

In [96]:
# Encode target for xgboost algo only

le = LabelEncoder()
risk_df["risk_score_encoded"] = le.fit_transform(risk_df["risk_score"])
print("Label Mapping: ", dict(zip(le.classes_, le.transform(le.classes_))))

Label Mapping:  {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}


In [97]:
risk_df = risk_df.sort_values("visit_date").reset_index(drop=True)
split_idx = int(len(risk_df)*0.8)

risk_train_xgb = risk_df.iloc[:split_idx].copy()
risk_test_xgb = risk_df.iloc[split_idx:].copy()

X_train_risk = risk_train_xgb[risk_features]
X_test_risk = risk_test_xgb[risk_features]

y_train_risk_xgb = risk_train_xgb["risk_score_encoded"]
y_test_risk_xgb = risk_test_xgb["risk_score_encoded"]

print(f"Train Shape: {X_train_risk.shape}")
print(f"Test Shape: {X_test_risk.shape}")

Train Shape: (20000, 13)
Test Shape: (5000, 13)


In [98]:
# Step 7 - XGBoost Model

risk_xgb_model = Pipeline(steps=[
    ("preprocessor", risk_preprocessor),
    ("classifier", XGBClassifier(
        objective = "multi:softmax",
        num_class =3,
        n_estimators = 500,
        max_depth = 6,
        learning_rate = 0.05,
        subsample = 0.8,
        colsample_bytree=0.8,
        random_state = 42,
        verbosity = 0
    ))
])

# Train the model
risk_xgb_model.fit(X_train_risk, y_train_risk_xgb)

# Predict
risk_xgb_pred_train = risk_xgb_model.predict(X_train_risk)
risk_xgb_pred_test = risk_xgb_model.predict(X_test_risk)

xgb_acc = accuracy_score(y_test_risk_xgb, risk_xgb_pred_test)
print(f"Risk XGB Train Accuracy :", round(accuracy_score(y_train_risk_xgb, risk_xgb_pred_train),4))
print(f"Risk XGB Test Accuracy :", round(accuracy_score(y_test_risk_xgb, risk_xgb_pred_test),4))
print(f"Risk XGB Test Weighted F1 :", round(f1_score(y_test_risk_xgb, risk_xgb_pred_test, average="weighted"),4))

print("Classification Report:\n")
print(classification_report(y_test_risk_xgb, risk_xgb_pred_test, target_names = le.classes_))
print("Confusion Matrix:\n", confusion_matrix(y_test_risk_xgb, risk_xgb_pred_test))

Risk XGB Train Accuracy : 0.9921
Risk XGB Test Accuracy : 0.9548
Risk XGB Test Weighted F1 : 0.9549
Classification Report:

              precision    recall  f1-score   support

        High       0.98      0.94      0.96       840
         Low       0.97      0.96      0.97      2360
      Medium       0.93      0.95      0.94      1800

    accuracy                           0.95      5000
   macro avg       0.96      0.95      0.95      5000
weighted avg       0.96      0.95      0.95      5000

Confusion Matrix:
 [[ 789    0   51]
 [   0 2275   85]
 [  13   77 1710]]


In [99]:
# Model Comparison
print("=" * 45)
print("MODEL A — RISK CLASSIFICATION COMPARISON")
print("=" * 45)
print(f"Logistic Regression : {baseline_acc:.4f}")
print(f"Random Forest       : {rf_acc:.4f}")
print(f"XGBoost             : {xgb_acc:.4f}")
print()
print(f"Best model: XGBoost → {max(baseline_acc, rf_acc, xgb_acc):.4f}")

MODEL A — RISK CLASSIFICATION COMPARISON
Logistic Regression : 0.9248
Random Forest       : 0.9294
XGBoost             : 0.9548

Best model: XGBoost → 0.9548


# Hyperparameter Tuning - Random Forest
* We use RandomizedSearchCV to find best RF parameters
* `f1_weighted` is our scoring metric coz if we consider accuracy, it may optimize for majority class

In [100]:
rf_param_grid = {
    "classifier__n_estimators":         [200,300,400],
    "classifier__max_depth":            [8,12,16,None],
    "classifier__min_samples_split":    [5,10,20],
    "classifier__min_samples_leaf":     [1,2,4]
}

rf_random_search = RandomizedSearchCV(
    risk_rf_model,
    param_distributions = rf_param_grid,
    n_iter = 20,
    cv = 3,
    scoring = "f1_weighted",
    n_jobs = -1,
    random_state = 42
)

rf_random_search.fit(X_train_risk, y_train_risk)
print("Best parameters: ", rf_random_search.best_params_)

Best parameters:  {'classifier__n_estimators': 200, 'classifier__min_samples_split': 20, 'classifier__min_samples_leaf': 2, 'classifier__max_depth': 16}


In [101]:
best_rf_model = rf_random_search.best_estimator_
rf_tuned_pred = best_rf_model.predict(X_test_risk)

print("Tuned RF Accuracy   :", round(accuracy_score(y_test_risk, rf_tuned_pred), 4))
print("Tuned RF Weighted F1:", round(f1_score(y_test_risk, rf_tuned_pred, average="weighted"), 4))
print("\nClassification Report:\n")
print(classification_report(y_test_risk, rf_tuned_pred))


Tuned RF Accuracy   : 0.4186
Tuned RF Weighted F1: 0.3736

Classification Report:

              precision    recall  f1-score   support

        High       0.14      0.04      0.07       838
         Low       0.46      0.70      0.56      2360
      Medium       0.35      0.23      0.28      1802

    accuracy                           0.42      5000
   macro avg       0.32      0.32      0.30      5000
weighted avg       0.37      0.42      0.37      5000



The hyperparameter tuning fails because it combines random values/rows but since our data is time series, it is mixing past and future data resulting in horrible evaluation metrics. We can look into TimeSeriesSplit from sklearn.model_selection but since our XGB model is already hitting 95%, we are content for now.

# Model B - Claim Outcome Prediction

**Business Purpose:** \
Predict whether an insurance claim will be Paid, Pending or Rejected - before submission \

**Target:** `claim_status` - Paid/Pending/Rejected \
**Split:** Time based on `billing_date`

**Leakage prevention:** \
`approved_amount` and  `payment_days` excluded - both are post outcome variables that are only known after the claim is processed 




In [102]:
# Step 1 - Define claim features

claim_target = "claim_status"
claim_features = [
    "age", 
    "gender", 
    "city", 
    "insurance_provider", 
    "chronic_flag", 
    "department", 
    "visit_type", 
    "length_of_stay_hours",
    "risk_score",
    "billed_amount",
    "days_since_registration", 
    "visit_frequency", 
    "avg_los_per_patient",
    "provider_rejection_rate", 
    "visit_month", 
    "visit_dayofweek",
    "high_cost_visit_flag"
]

claim_df = df[claim_features + [claim_target, "billing_date"]].copy()
claim_df = claim_df.dropna(subset =[claim_target, "billing_date"])
print(f"Claim dataset shape : {claim_df.shape}")
claim_df.head()

Claim dataset shape : (25000, 19)


,age,gender,city,insurance_provider,chronic_flag,department,visit_type,length_of_stay_hours,risk_score,billed_amount,days_since_registration,visit_frequency,avg_los_per_patient,provider_rejection_rate,visit_month,visit_dayofweek,high_cost_visit_flag,claim_status,billing_date
0,15,F,Mumbai,CareOne,0,General,OPD,9.63,Low,9612.77,5,4,21.120000,0.256876,1,3,0,Rejected,2026-01-19
1,3,M,Bangalore,CareOne,0,ICU,ICU,59.60,High,88539.01,141,8,23.750000,0.256876,1,3,1,Rejected,2026-01-05
2,44,M,Pune,MediCareX,1,ICU,ER,59.28,High,88539.01,165,3,32.460000,0.242556,1,3,1,Pending,2026-01-20
3,47,F,Delhi,CareOne,1,General,OPD,25.15,Medium,20958.52,60,3,30.056667,0.256876,1,3,0,Paid,2026-01-15
4,14,F,Chennai,SecureLife,1,Cardiology,ER,42.88,Medium,74921.48,330,9,29.030000,0.157496,1,3,1,Paid,2026-01-16


In [103]:
# Claim status distribution
print("Claim Status Distribution")
print(claim_df[claim_target].value_counts())
print()

print("Claim % Status Distribution")
print((claim_df[claim_target].value_counts(normalize=True)*100).round(2))

Claim Status Distribution
claim_status
Paid        13638
Pending      6096
Rejected     5266
Name: count, dtype: int64

Claim % Status Distribution
claim_status
Paid        54.55
Pending     24.38
Rejected    21.06
Name: proportion, dtype: float64


# Step 3 - Time based split for claim model 
Sort by `billing_date` as billing happens after visit.


In [104]:
claim_df = claim_df.sort_values("billing_date").reset_index(drop=True)
split_idx = int(len(claim_df)*0.8)

claim_train = claim_df.iloc[: split_idx].copy()
claim_test = claim_df.iloc[split_idx : ].copy()

X_train_claim = claim_train[claim_features]
X_test_claim = claim_test[claim_features]

y_train_claim = claim_train[claim_target]
y_test_claim = claim_test[claim_target]

print(f"Train Shape: {X_train_claim.shape}")
print(f"Test Shape: {X_test_claim.shape}")
print(f"Train period : {claim_train['billing_date'].min().date()} -> {claim_train['billing_date'].max().date()}")
print(f"Test period : {claim_test['billing_date'].min().date()} -> {claim_test['billing_date'].max().date()}")

Train Shape: (20000, 17)
Test Shape: (5000, 17)
Train period : 2025-01-28 -> 2026-01-17
Test period : 2026-01-17 -> 2026-01-20


# Preprocessing 

In [105]:
claim_numeric_features =[f for f in claim_features if df[f].dtype != "object"]
claim_categorical_features = [f for f in claim_features if f not in claim_numeric_features]

print("claim numeric", claim_numeric_features)
print("claim_categorical", claim_categorical_features)

claim numeric ['age', 'chronic_flag', 'length_of_stay_hours', 'billed_amount', 'days_since_registration', 'visit_frequency', 'avg_los_per_patient', 'provider_rejection_rate', 'visit_month', 'visit_dayofweek', 'high_cost_visit_flag']
claim_categorical ['gender', 'city', 'insurance_provider', 'department', 'visit_type', 'risk_score']


In [106]:
claim_numeric_transformer = Pipeline(steps = [
    ("imputer", SimpleImputer(strategy = "median"))
])

claim_categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy = "most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown = "ignore"))
])

claim_preprocessor = ColumnTransformer(transformers=[
    ("num", claim_numeric_transformer, claim_numeric_features),
    ("cat", claim_categorical_transformer, claim_categorical_features)
])

print("Claim Preprocessing Done ✔")

Claim Preprocessing Done ✔


# Modeling - Baseline

In [107]:
#  Logistic Regression Model

claim_baseline_model = Pipeline(steps=[
    ("preprocessor", claim_preprocessor),
    ("classifier", LogisticRegression(
        max_iter = 1000,
        class_weight = "balanced"
    ))
])

# Train
claim_baseline_model.fit(X_train_claim, y_train_claim)

# Predict
y_train_pred = claim_baseline_model.predict(X_train_claim)
y_test_pred = claim_baseline_model.predict(X_test_claim)

# Evaluate
baseline_accuracy = accuracy_score(y_test_pred, y_test_claim)

print("Claim Baseline Train Accuracy :", round(accuracy_score(y_train_claim, y_train_pred), 4))
print("Claim Baseline Test Accuracy  :", round(accuracy_score(y_test_claim,  y_test_pred), 4))
print("Claim Baseline Test Weighted F1:", round(f1_score(y_test_claim, y_test_pred, average="weighted"), 4))
print("\nClassification Report:\n")
print(classification_report(y_test_claim, y_test_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test_claim, y_test_pred))

Claim Baseline Train Accuracy : 0.4761
Claim Baseline Test Accuracy  : 0.387
Claim Baseline Test Weighted F1: 0.4022

Classification Report:

              precision    recall  f1-score   support

        Paid       0.63      0.42      0.50      2771
     Pending       0.26      0.22      0.24      1189
    Rejected       0.24      0.50      0.33      1040

    accuracy                           0.39      5000
   macro avg       0.38      0.38      0.35      5000
weighted avg       0.46      0.39      0.40      5000

Confusion Matrix:
 [[1154  548 1069]
 [ 372  264  553]
 [ 305  218  517]]


# Random Forest on Claim's Data

In [108]:
# Step 6 - Random Forest Model

claim_rf_model = Pipeline(steps=[
    ("preprocessor", claim_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators = 300,
        max_depth = 16,
        min_samples_split = 20,
        class_weight = "balanced",
        random_state = 42
    ))
])

# Train the model
claim_rf_model.fit(X_train_claim, y_train_claim)

# Predict
claim_rf_pred_train = claim_rf_model.predict(X_train_claim)
claim_rf_pred_test = claim_rf_model.predict(X_test_claim)

rf_acc = accuracy_score(y_test_claim, claim_rf_pred_test)
print(f"Claim RF Train Accuracy :", round(accuracy_score(y_train_claim, claim_rf_pred_train),4))
print(f"Claim RF Test Accuracy :", round(accuracy_score(y_test_claim, claim_rf_pred_test),4))
print(f"Claim RF Test Weighted F1 :", round(f1_score(y_test_claim, claim_rf_pred_test, average="weighted"),4))

print("Classification Report:\n")
print(classification_report(y_test_claim, claim_rf_pred_test))
print("Confusion Matrix:\n", confusion_matrix(y_test_claim, claim_rf_pred_test))

Claim RF Train Accuracy : 0.7974
Claim RF Test Accuracy : 0.4936
Claim RF Test Weighted F1 : 0.487
Classification Report:

              precision    recall  f1-score   support

        Paid       0.63      0.67      0.65      2771
     Pending       0.31      0.23      0.26      1189
    Rejected       0.29      0.34      0.31      1040

    accuracy                           0.49      5000
   macro avg       0.41      0.41      0.41      5000
weighted avg       0.48      0.49      0.49      5000

Confusion Matrix:
 [[1847  394  530]
 [ 599  270  320]
 [ 472  217  351]]


# XGBoost for Claim

In [109]:
le_claim = LabelEncoder()
claim_df["claim_status_encoded"] = le_claim.fit_transform(claim_df["claim_status"])
print(f"Label Mapping: {dict(zip(le_claim.classes_, le_claim.transform(le_claim.classes_)))}")

Label Mapping: {'Paid': np.int64(0), 'Pending': np.int64(1), 'Rejected': np.int64(2)}


In [110]:
claim_df = claim_df.sort_values("billing_date").reset_index(drop=True)
split_idx = int(len(claim_df)*0.8)

claim_train_xgb = claim_df.iloc[: split_idx].copy()
claim_test_xgb = claim_df.iloc[split_idx : ].copy()

X_train_claim = claim_train_xgb[claim_features]
X_test_claim = claim_test_xgb[claim_features]

y_train_claim_xgb = claim_train_xgb["claim_status_encoded"]
y_test_claim_xgb = claim_test_xgb["claim_status_encoded"]

print(f"Train Shape: {X_train_claim.shape}")
print(f"Test Shape: {X_test_claim.shape}")
print(f"Train period : {claim_train_xgb['billing_date'].min().date()} -> {claim_train_xgb['billing_date'].max().date()}")
print(f"Test period : {claim_test_xgb['billing_date'].min().date()} -> {claim_test_xgb['billing_date'].max().date()}")


Train Shape: (20000, 17)
Test Shape: (5000, 17)
Train period : 2025-01-28 -> 2026-01-17
Test period : 2026-01-17 -> 2026-01-20


In [111]:
claim_xgb_model = Pipeline(steps=[
    ("preprocessor", claim_preprocessor),
    ("classifier", XGBClassifier(
        objective = "multi:softmax",
        num_class =3,
        n_estimators = 500,
        max_depth = 6,
        learning_rate = 0.05,
        subsample = 0.8,
        colsample_bytree=0.8,
        random_state = 42,
        verbosity = 0
    ))
])

# Train the model
claim_xgb_model.fit(X_train_claim, y_train_claim_xgb)

# Predict
claim_xgb_pred_train = claim_xgb_model.predict(X_train_claim)
claim_xgb_pred_test = claim_xgb_model.predict(X_test_claim)

claim_xgb_acc = accuracy_score(y_test_claim_xgb, claim_xgb_pred_test)
print(f"Claim XGB Train Accuracy :", round(accuracy_score(y_train_claim_xgb, claim_xgb_pred_train),4))
print(f"Claim XGB Test Accuracy :", round(accuracy_score(y_test_claim_xgb, claim_xgb_pred_test),4))
print(f"Claim XGB Test Weighted F1 :", round(f1_score(y_test_claim_xgb, claim_xgb_pred_test, average="weighted"),4))

print("Classification Report:\n")
print(classification_report(y_test_claim_xgb, claim_xgb_pred_test, target_names = le_claim.classes_))
print("Confusion Matrix:\n", confusion_matrix(y_test_claim_xgb, claim_xgb_pred_test))

Claim XGB Train Accuracy : 0.7531
Claim XGB Test Accuracy : 0.5388
Claim XGB Test Weighted F1 : 0.4706
Classification Report:

              precision    recall  f1-score   support

        Paid       0.59      0.87      0.70      2773
     Pending       0.31      0.11      0.16      1188
    Rejected       0.31      0.15      0.20      1039

    accuracy                           0.54      5000
   macro avg       0.40      0.38      0.36      5000
weighted avg       0.47      0.54      0.47      5000

Confusion Matrix:
 [[2406  168  199]
 [ 906  131  151]
 [ 765  117  157]]


In [112]:
# Model Comparison
print("=" * 45)
print("MODEL B — CLAIM OUTCOME COMPARISON")
print("=" * 45)
print(f"Logistic Regression : {baseline_accuracy:.4f}")
print(f"Random Forest       : {rf_acc:.4f}")
print(f"XGBoost             : {claim_xgb_acc:.4f}")
print()
print(f"Best model: XGBoost → {max(baseline_accuracy, rf_acc, claim_xgb_acc):.4f}")

MODEL B — CLAIM OUTCOME COMPARISON
Logistic Regression : 0.3870
Random Forest       : 0.4936
XGBoost             : 0.5388

Best model: XGBoost → 0.5388


# Save Model Artifacts
We save both the model using joblib.

In [113]:
joblib.dump(risk_rf_model, "../models/risk_model_complete_pipeline.joblib")
joblib.dump(claim_rf_model, "../models/claim_model_complete_pipeline.joblib")

print("Models saved:")
print(" ✔ ../models/risk_model_complete_pipeline.joblib")
print(" ✔ ../models/claim_model_complete_pipeline.joblib")

Models saved:
 ✔ ../models/risk_model_complete_pipeline.joblib
 ✔ ../models/claim_model_complete_pipeline.joblib


# Save Feature schema
The feature schema tells our FastAPI service: 
* Which features to expect in the request payload
* Which column is the target
* What split strategy was used

This is a contract between model and API

In [114]:
feature_schema = {
    "risk_model_features"   : risk_features,     # list of 13 features
    "claim_model_features"  : claim_features,    # list 17 features
    "risk_target"           : "risk_score",      # what model A predicts
    "claim_target"          : "claim_status",    # what model B predicts
    "risk_time_column"      : "visit_date",      # split column for model A
    "claim_time_column"     : "billing_date",    # split column for model B
    "split_strategy"        : "earliest 80 percent train, latest 20 percent test"        
}

with open("../outputs/feature_schema.json", "w") as f:
    json.dump(feature_schema, f, indent=4)

print("Feature schema saved ✔")
print(" --> ../outputs/feature_schema.json")

Feature schema saved ✔
 --> ../outputs/feature_schema.json
